### Data ingestion to vector db

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\RAKPOOJA\Downloads\AI\RAG\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
##Read all the pdfs inside the directory
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")
print(all_pdf_documents)


Found 1 PDF files to process

Processing: ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf


Ignoring wrong pointing object 13 0 (offset 0)


  ✓ Loaded 3 pages

Total documents loaded: 3
[Document(metadata={'producer': 'macOS Version 13.4 (Build 22F66) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20240324062134Z00'00'", 'title': 'MOHAMMED_ASHIQ_J2_V02', 'moddate': "D:20240324062134Z00'00'", 'source': '..\\data\\pdf\\ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf', 'file_type': 'pdf'}, page_content='ASHIQ U \nDevOps Engineer \n+91-7907558767\nashiqummathoor@outlook.com\nLinkedIn\nOBJECTIVE\nSUMMARY\nEnthusiastic about tackling challenging roles and collaborating with diverse teams to drive \norganisational success. Possessing extensive experience across various cloud and on-premises \nenvironments, I am dedicated to implementing high availability and resilient architectures. With \na clear, logical mindset and a practical approach to problem-solving, I strive to see projects \nthrough to successful completion, contributing eﬀec

In [3]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    # if split_docs:
    #     print(f"\nExample chunk:")
    #     print(f"Content: {split_docs[0].page_content[:200]}...")
    #     print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
    
chunks=split_documents(all_pdf_documents)
print("CHHH",all_pdf_documents)
print("CHHH",chunks)


Split 3 documents into 10 chunks
CHHH [Document(metadata={'producer': 'macOS Version 13.4 (Build 22F66) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20240324062134Z00'00'", 'title': 'MOHAMMED_ASHIQ_J2_V02', 'moddate': "D:20240324062134Z00'00'", 'source': '..\\data\\pdf\\ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf', 'file_type': 'pdf'}, page_content='ASHIQ U \nDevOps Engineer \n+91-7907558767\nashiqummathoor@outlook.com\nLinkedIn\nOBJECTIVE\nSUMMARY\nEnthusiastic about tackling challenging roles and collaborating with diverse teams to drive \norganisational success. Possessing extensive experience across various cloud and on-premises \nenvironments, I am dedicated to implementing high availability and resilient architectures. With \na clear, logical mindset and a practical approach to problem-solving, I strive to see projects \nthrough to successful completion, contributing eﬀectively t

embedding And vectorStoreDB

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = r"C:\models\all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise



## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager
vectorstore=VectorStore()
vectorstore
chunks
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        print(query_embedding)
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            print(results)

            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    # if similarity_score >= score_threshold:
                    retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                    })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever.retrieve("")


Loading embedding model: C:\models\all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: C:\models\all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 10
Generating embeddings for 10 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (10, 384)
Adding 10 documents to vector store...
Successfully added 10 documents to vector store
Total documents in collection: 20
Retrieving documents for query: 'TECHNICAL SKILLS'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
[-4.54234481e-02 -3.15104909e-02  3.09521724e-02  2.74984296e-02
 -1.09237000e-01 -7.37487078e-02  1.55513749e-01  8.64529163e-02
 -4.29283530e-02  2.81171985e-02 -3.61163765e-02 -3.82693075e-02
 -1.87339857e-02  2.26403680e-02 -5.32017043e-03 -3.66274603e-02
 -6.26827008e-04 -2.85426099e-02 -4.54834215e-02 -1.22435085e-01
 -1.52174924e-02  6.64349943e-02  3.86510529e-02 -3.79773006e-02
 -1.52553758e-02 -1.95384007e-02  1.55987097e-02 -1.43372240e-02
  4.21346873e-02 -6.68794438e-02 -4.86441851e-02  3.97745892e-02
 -4.03083265e-02  5.79961538e-02 -4.00300808e-02  4.27055396e-02
  1.14839571e-02  5.52043505e-02  4.71654087e-02 -5.34551814e-02
 -4.56606299e-02 -7.83631504e-02  3.51680554e-02 -7.14614019e-02
  4.12459187e-02 -1.76171288e-02  2.08237302e-02 -3.73817310e-02
  1.19962897e-02  1.14060612e-02 -5.74287102e-02 -6.88321469e-03
 -1.93096045e-03 -2.40279194e-02  6.64291680e-02 -4.13624989e-03
  6.54040128e-02  5.27433082e-02 -4.15334525e-03

[{'id': 'doc_276d0cb5_9',
  'content': '๏Worked extensively with GoCD pipeline, facilitating customer migration from cloud vendors \nand promoting cloud-agnostic practices to enhance operational ﬂexibility and reduce \ndependency.\nEDUCATIONAL QUALIFICATIONS\n๏B. Tech Computer Science and Engineering 2015 - 2019  \n๏Higher Secondary (Kerala State Board) – 2013  \n๏Secondary School (Kerala State Board SSLC) – 2012 \u2028\nDECLARATION\n     I am keen to continue my career and prepared to work hard to achieve my organization \nobjectives and I hereby declare that the information furnished above is true to the best of my \nknowledge.',
  'metadata': {'total_pages': 3,
   'source': '..\\data\\pdf\\ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf',
   'file_type': 'pdf',
   'creationdate': "D:20240324062134Z00'00'",
   'page': 2,
   'doc_index': 9,
   'producer': 'macOS Version 13.4 (Build 22F66) Quartz PDFContext',
   'title': 'MOHAMMED_ASHIQ_J2_V02',
   'source_file': 'ASHIQ_DEVOPS_ENGINEER_J2_V02.pdf',
 